# Otondo · Nigerian voices with Gemini TTS (free tier)

**Every day, just do: Runtime → Run all.** It asks for your key, makes as many lines as today's free limit allows,
checks them with Whisper, and downloads `nysc_voices_gemini.zip` at the end. Send that zip to Claude.

- Free key: **https://aistudio.google.com/apikey**
- Progress is saved in Google Drive (`MyDrive/nysc_voices_gemini`), so tomorrow it continues where it stopped.
- If Colab disconnects, you must start again from Step 1 (Run all does that for you).

In [ ]:
# Step 1 · setup (no installs needed: talks to Gemini directly, so library versions can't break it)
import os, json, time, wave, re, io, base64, subprocess, numpy as np, requests
print('ok')

In [ ]:
# Step 2 · your key + where to save (Google Drive keeps progress between days)
from getpass import getpass
API_KEY = getpass('Paste your Gemini API key: ').strip()
BASE = 'https://generativelanguage.googleapis.com/v1beta'
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/nysc_voices_gemini'; os.makedirs(OUT, exist_ok=True)
# the line list comes straight from the game repo
SCRIPT_URL = 'https://raw.githubusercontent.com/Auspicious14/nysc-survival/main/game3d-src/voice/script.json'
LINES = requests.get(SCRIPT_URL, timeout=60).json()
# lines marked "redo" in the script (old or faulty takes) are deleted once, so they get remade
rd = f'{OUT}/.redo_done'; redone = set(open(rd).read().split()) if os.path.exists(rd) else set()
for l in LINES:
    if l.get('redo') and l['id'] not in redone:
        if os.path.exists(f"{OUT}/{l['id']}.mp3"): os.remove(f"{OUT}/{l['id']}.mp3")
        redone.add(l['id'])
open(rd, 'w').write(' '.join(sorted(redone)))
done = {f[:-4] for f in os.listdir(OUT) if f.endswith('.mp3')}
print(len(LINES), 'lines in the game ·', len(done), 'already done')
# find a TTS model this key can use
r = requests.get(f'{BASE}/models', params={'key': API_KEY, 'pageSize': 200}, timeout=60); r.raise_for_status()
names = [m['name'].split('/')[-1] for m in r.json().get('models', []) if 'tts' in m['name']]
MODEL = next((n for n in ['gemini-2.5-flash-preview-tts', 'gemini-2.5-flash-tts', 'gemini-2.5-pro-preview-tts'] if n in names), names[0] if names else 'gemini-2.5-flash-preview-tts')
print('TTS models:', names, '→ using', MODEL)

In [ ]:
# Step 3 · helpers: one request → PCM → mp3; batches split on long silences
EXHAUSTED = set()
def tts(prompt, voice):
    global MODEL
    body = {'contents': [{'parts': [{'text': prompt}]}],
            'generationConfig': {'responseModalities': ['AUDIO'],
                                 'speechConfig': {'voiceConfig': {'prebuiltVoiceConfig': {'voiceName': voice}}}}}
    for attempt in range(8):
        try:
            r = requests.post(f'{BASE}/models/{MODEL}:generateContent', params={'key': API_KEY}, json=body, timeout=180)
        except Exception as e:
            print('   network:', str(e)[:120]); time.sleep(8); continue
        if r.status_code == 200:
            part = r.json()['candidates'][0]['content']['parts'][0]
            return np.frombuffer(base64.b64decode(part['inlineData']['data']), dtype=np.int16)
        msg = r.text
        if r.status_code == 429:
            if 'PerDay' in msg or 'per day' in msg.lower() or attempt == 3:
                EXHAUSTED.add(MODEL); nxt = [n for n in names if n not in EXHAUSTED]
                if not nxt: raise RuntimeError('DAILY_LIMIT on every TTS model')
                MODEL = nxt[0]; print('   daily limit reached → switching to', MODEL); continue
            print('   rate limit, waiting 65 s…'); time.sleep(65); continue
        print(f'   error {r.status_code}:', msg[:300])
        if r.status_code in (400, 403, 404) and len(names) > 1:  # this model can't do it → try the next
            EXHAUSTED.add(MODEL); nxt = [n for n in names if n not in EXHAUSTED]
            if not nxt: raise RuntimeError('No TTS model works for this key')
            MODEL = nxt[0]; print('   switching to', MODEL); continue
        time.sleep(8)
    raise RuntimeError('FAILED')
def save_mp3(pcm, path):
    wav = path[:-4] + '.wav'
    with wave.open(wav, 'wb') as w: w.setnchannels(1); w.setsampwidth(2); w.setframerate(24000); w.writeframes(pcm.tobytes())
    subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', wav, '-af', 'silenceremove=start_periods=1:start_threshold=-45dB,areverse,silenceremove=start_periods=1:start_threshold=-45dB,areverse,apad=pad_dur=0.12,loudnorm=I=-16:TP=-1.5', '-ar', '24000', '-ac', '1', '-b:a', '96k', path], check=True)
    os.remove(wav)
def split_silence(pcm, n, min_sil=0.75):
    fr = 480  # 20 ms
    k = len(pcm) // fr; rms = np.sqrt((pcm[:k * fr].astype(np.float32).reshape(k, fr) ** 2).mean(1))
    quiet = rms < max(220, np.percentile(rms, 20) * 1.6)
    runs, s = [], None
    for i, q in enumerate(quiet):
        if q and s is None: s = i
        if not q and s is not None:
            if (i - s) * 0.02 >= min_sil and s > 0: runs.append((s, i))
            s = None
    runs = sorted(runs, key=lambda r: r[1] - r[0], reverse=True)[:n - 1]
    cuts = sorted(((a + b) // 2) * fr for a, b in runs)
    if len(cuts) != n - 1: return None
    return np.split(pcm, cuts)
def prompt_for(group):
    notes = f"### DIRECTOR'S NOTES (never read these aloud)\n{group[0]['style']}\nSpeak ONLY the transcript lines. Do not read numbers in front of lines, headings or these notes."
    if len(group) == 1: return f"{notes}\n\n### TRANSCRIPT\n{group[0]['text']}"
    body = '\n'.join(l['text'] for l in group)
    return f"{notes}\nThere are {len(group)} separate lines. After each line stay completely silent for two full seconds.\n\n### TRANSCRIPT\n{body}"
print('ready')

In [ ]:
# Step 4 · samples (off by default). Set SAMPLES = True to hear six voices first; they are saved as real game lines.
SAMPLES = False
import glob, shutil
from IPython.display import Audio, display
for s in glob.glob('/content/sample_*.mp3'):  # keep samples from an earlier run
    shutil.move(s, f"{OUT}/{os.path.basename(s)[7:]}")
have = {f[:-4] for f in os.listdir(OUT) if f.endswith('.mp3')}
for want in (['Fenrir', 'Charon', 'Kore', 'Puck', 'Sulafat', 'Orus'] if SAMPLES else []):
    l = next((x for x in LINES if x['voice'] == want and x['prio'] == 1), None)
    if not l: continue
    p = f"{OUT}/{l['id']}.mp3"
    if l['id'] not in have: save_mp3(tts(prompt_for([l]), l['voice']), p)
    print(l['voice'], '·', l['text']); display(Audio(p))


In [ ]:
# Step 5 · generate (stops by itself at today's free limit; Run all again tomorrow)
if 'LINES' not in globals(): raise SystemExit('Run the notebook from the top: Runtime → Run all.')
BATCH = 8
todo = [l for l in LINES if l['id'] not in {f[:-4] for f in os.listdir(OUT)}]
print(len(todo), 'lines to go')
groups, cur = [], []
for l in todo:  # same voice + same style per request
    if cur and (cur[0]['voice'] != l['voice'] or cur[0]['style'] != l['style'] or len(cur) >= BATCH): groups.append(cur); cur = []
    cur.append(l)
if cur: groups.append(cur)
made = 0
try:
    for gi, g in enumerate(groups):
        pcm = tts(prompt_for(g), g[0]['voice'])
        parts = [pcm] if len(g) == 1 else split_silence(pcm, len(g))
        if parts is None:  # could not split cleanly → try once more, then halves
            pcm = tts(prompt_for(g), g[0]['voice']); parts = split_silence(pcm, len(g))
        if parts is None:
            h = len(g) // 2; parts = []
            for sub in (g[:h], g[h:]):
                p2 = tts(prompt_for(sub), sub[0]['voice']); sp = [p2] if len(sub) == 1 else split_silence(p2, len(sub))
                parts += sp if sp else [tts(prompt_for([l]), l['voice']) for l in sub]
        for l, part in zip(g, parts): save_mp3(part, f"{OUT}/{l['id']}.mp3"); made += 1
        print(f"{gi + 1}/{len(groups)} · {g[0]['voice']} · {made} new files")
        time.sleep(7)  # stay under the per-minute limit
    print('All done ✅')
except RuntimeError as e:
    print(f"Stopped: {e}. {made} new files saved to Drive. Tomorrow: Runtime → Run all again.")

In [ ]:
# Step 6 · fix + check (no Gemini quota used). Only new files are checked. Listens to every file with Whisper, cuts each line exactly,
#     removes anything extra (e.g. the model reading instructions aloud) and deletes bad lines so step 5 redoes them tomorrow.
!pip -q install faster-whisper rapidfuzz
import numpy as np, subprocess, re, unicodedata, torch
from faster_whisper import WhisperModel
from rapidfuzz import fuzz
wm = WhisperModel('small.en', device='cuda' if torch.cuda.is_available() else 'cpu', compute_type='float16' if torch.cuda.is_available() else 'int8')
SR = 16000
def load(p): return np.frombuffer(subprocess.run(['ffmpeg', '-v', 'error', '-i', p, '-ac', '1', '-ar', str(SR), '-f', 's16le', '-'], capture_output=True).stdout, np.int16).astype(np.float32) / 32768
def norm(w): w = unicodedata.normalize('NFKD', w.lower()); return re.sub(r'[^a-z0-9]', '', w)
def words(t): return [x for x in (norm(w) for w in re.split(r'[\s\-–—/…]+', t.replace(',000', '000'))) if x]
have = {f[:-4] for f in os.listdir(OUT) if f.endswith('.mp3')}
ck = f'{OUT}/.checked'; checked = set(open(ck).read().split()) if os.path.exists(ck) else set()
have -= checked
byv = {}
for l in LINES:
    if l['id'] in have: byv.setdefault(l['voice'], []).append(l)
kept, redo = 0, []
for v, ls in byv.items():
    gap = np.zeros(int(SR * 0.8), np.float32)
    audio = np.concatenate([x for l in ls for x in (load(f"{OUT}/{l['id']}.mp3"), gap)])
    segs, _ = wm.transcribe(audio, language='en', word_timestamps=True, beam_size=5, vad_filter=False,
                            initial_prompt='Corper, NYSC, naira, abeg, omo, LGI, CDS, Ikeja, Platoon 5, khaki, ehn, o.')
    hw = [(norm(w.word), w.start, w.end) for s in segs for w in s.words if norm(w.word)]
    tw = [(w, j) for j, l in enumerate(ls) for w in words(l['text'])]
    n, m = len(hw), len(tw)
    # global alignment (heard words vs script words); skipping heard words is cheap so extra speech is dropped
    D = np.zeros((n + 1, m + 1)); B = np.zeros((n + 1, m + 1), np.int8)
    D[0, 1:] = np.arange(1, m + 1) * 1.0; B[0, 1:] = 2; D[1:, 0] = np.arange(1, n + 1) * 0.6; B[1:, 0] = 1
    for i in range(1, n + 1):
        for k in range(1, m + 1):
            s = fuzz.ratio(hw[i - 1][0], tw[k - 1][0])
            c = (D[i - 1, k - 1] + (0 if s >= 70 else 1.4), 0)
            c = min(c, (D[i - 1, k] + 0.6, 1), (D[i, k - 1] + 1.0, 2))
            D[i, k], B[i, k] = c
    i, k, match = n, m, {}
    while i > 0 or k > 0:
        b = B[i, k]
        if b == 0:
            if fuzz.ratio(hw[i - 1][0], tw[k - 1][0]) >= 70: match.setdefault(tw[k - 1][1], []).append(i - 1)
            i, k = i - 1, k - 1
        elif b == 1: i -= 1
        else: k -= 1
    for j, l in enumerate(ls):
        nw = sum(1 for w in tw if w[1] == j); got = sorted(match.get(j, []))
        if not got or len(got) / nw < 0.6:
            redo.append(l); os.remove(f"{OUT}/{l['id']}.mp3"); continue
        a = max(0, hw[got[0]][1] - 0.12); b = hw[got[-1]][2] + 0.25
        seg = (audio[int(a * SR):int(b * SR)] * 32767).astype(np.int16)
        wav = f"/content/_{l['id']}.wav"
        with wave.open(wav, 'wb') as w: w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR); w.writeframes(seg.tobytes())
        subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', wav, '-af', 'apad=pad_dur=0.1,loudnorm=I=-16:TP=-1.5', '-ar', '24000', '-ac', '1', '-b:a', '96k', f"{OUT}/{l['id']}.mp3"], check=True)
        os.remove(wav); kept += 1
    print(f'{v}: {len(ls)} lines checked')
open(ck, 'w').write(' '.join(sorted(checked | {l['id'] for ls in byv.values() for l in ls if os.path.exists(f"{OUT}/{l['id']}.mp3")})))
print(f'\n✅ {kept} lines cut cleanly · ❌ {len(redo)} removed (step 5 redoes them tomorrow):')
for l in redo: print('  ', l['voice'], '·', l['text'][:70])


In [ ]:
# Step 7 · download the zip for Claude (works with whatever is finished so far)
import shutil
ids = sorted(f[:-4] for f in os.listdir(OUT) if f.endswith('.mp3'))
json.dump(ids, open(f'{OUT}/manifest.json', 'w'))
shutil.make_archive('/content/nysc_voices_gemini', 'zip', OUT)
print(len(ids), 'voice files zipped')
from google.colab import files; files.download('/content/nysc_voices_gemini.zip')